<a href="https://colab.research.google.com/github/dhavanavijaywork/GenAI_Dhavana/blob/main/T6_PDF_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip -q install pypdf sentence-transformers faiss-cpu google-genai gradio

In [17]:
import os
import re
import numpy as np
import faiss
import gradio as gr

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from google import genai
from google.genai import types

In [18]:
from getpass import getpass

GEMINI_API_KEY = getpass("Enter your Gemini API Key: ")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini API configured successfully!")

Enter your Gemini API Key: ··········
Gemini API configured successfully!


In [19]:
print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded!")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!


In [20]:
pages = []
chunks = []
chunk_embeddings = None
faiss_index = None

document_name = ""
total_pages = 0
total_chunks = 0

In [21]:
def clean_text(text):
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def create_chunks(text, page_number, chunk_size=500, overlap=100):
    words = text.split()

    page_chunks = []

    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))

        chunk_text = " ".join(words[start:end])

        if chunk_text.strip():
            page_chunks.append({
                "text": chunk_text,
                "page": page_number
            })

        if end >= len(words):
            break

        start = end - overlap

    return page_chunks


def process_pdf(file):
    global pages
    global chunks
    global chunk_embeddings
    global faiss_index
    global document_name
    global total_pages
    global total_chunks

    if file is None:
        return "Please upload a PDF."

    document_name = os.path.basename(file.name)

    reader = PdfReader(file.name)

    total_pages = len(reader.pages)

    pages = []
    chunks = []

    # Extract text page by page
    for page_number, page in enumerate(reader.pages, start=1):

        text = page.extract_text()

        if text is None:
            text = ""

        text = clean_text(text)

        pages.append({
            "page": page_number,
            "text": text
        })

        if text:
            page_chunks = create_chunks(
                text,
                page_number,
                chunk_size=500,
                overlap=100
            )

            chunks.extend(page_chunks)

    total_chunks = len(chunks)

    if total_chunks == 0:
        return (
            f"Document: {document_name}\n"
            f"Pages: {total_pages}\n\n"
            "No extractable text was found in this PDF."
        )

    # Create embeddings
    texts = [chunk["text"] for chunk in chunks]

    chunk_embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    # Create FAISS index
    dimension = chunk_embeddings.shape[1]

    faiss_index = faiss.IndexFlatIP(dimension)

    faiss_index.add(
        chunk_embeddings.astype("float32")
    )

    return (
        f"## Document Loaded Successfully\n\n"
        f"**Document:** `{document_name}`\n\n"
        f"📄 **Number of pages:** {total_pages}\n\n"
        f"🧩 **Number of chunks:** {total_chunks}\n\n"
        f"📐 **Chunk size:** 500 words\n\n"
        f"🔄 **Chunk overlap:** 100 words\n\n"
        f"🔎 **Retrieval:** FAISS + cosine similarity"
    )

In [22]:
def retrieve_chunks(question, top_k=5):

    if faiss_index is None:
        return []

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = faiss_index.search(
        question_embedding.astype("float32"),
        min(top_k, len(chunks))
    )

    retrieved = []

    for score, index in zip(scores[0], indices[0]):

        if index == -1:
            continue

        retrieved.append({
            "text": chunks[index]["text"],
            "page": chunks[index]["page"],
            "score": float(score),
            "chunk_index": int(index)
        })

    return retrieved

In [23]:
def generate_answer(question, retrieved_chunks):

    if not retrieved_chunks:
        return "I could not find relevant information in the uploaded document."

    context_parts = []

    for item in retrieved_chunks:
        context_parts.append(
            f"""
[Chunk {item['chunk_index']} | PDF Page {item['page']}]

{item['text']}
"""
        )

    context = "\n".join(context_parts)

    prompt = f"""
You are a document question-answering assistant.

Answer the user's question ONLY using the retrieved
content from the uploaded document.

Rules:
1. Do not use outside knowledge.
2. Do not invent information.
3. If the answer is not present in the retrieved context,
   say: "I could not find this information in the uploaded document."
4. Mention the relevant PDF page number when possible.
5. Give a clear and concise answer.

USER QUESTION:
{question}

RETRIEVED DOCUMENT CONTEXT:
{context}

ANSWER:
"""

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )

    return response.text

In [24]:
def ask_question(question, top_k):

    if faiss_index is None:
        return "Please upload a PDF first.", ""

    if not question.strip():
        return "Please enter a question.", ""

    retrieved = retrieve_chunks(question, int(top_k))

    answer = generate_answer(question, retrieved)

    retrieval_output = "## 🔎 Retrieved Chunks\n\n"

    for rank, item in enumerate(retrieved, start=1):
        retrieval_output += (
            f"### {rank}. Chunk {item['chunk_index']}\n"
            f"**PDF Page:** {item['page']}  \n"
            f"**Similarity Score:** {item['score']:.4f}\n\n"
            f"> {item['text']}\n\n"
            f"---\n\n"
        )

    return answer, retrieval_output

In [25]:
def show_all_chunks():
    if not chunks:
        return "No chunks available. Please upload and process a PDF first."

    output = "## All Chunks in Document\n\n"
    for i, chunk in enumerate(chunks):
        output += (
            f"### Chunk {i}\n"
            f"**PDF Page:** {chunk['page']}\n\n"
            f"> {chunk['text']}\n\n"
            f"---\n\n"
        )
    return output

In [26]:
def show_all_chunks():
    if not chunks:
        return "No chunks available. Please upload and process a PDF first."

    output = "## All Chunks in Document\n\n"
    for i, chunk in enumerate(chunks):
        output += (
            f"### Chunk {i}\n"
            f"**PDF Page:** {chunk['page']}\n\n"
            f"> {chunk['text']}\n\n"
            f"---\n\n"
        )
    return output

with gr.Blocks(title="PDF RAG System") as demo:

    gr.Markdown("""
    # 📚 PDF RAG SYSTEM

    ### Upload a PDF → Create Chunks → Retrieve → Ask Questions

    **PDF → Text Extraction → Chunking → Embeddings → FAISS → Retrieval → Gemini**
    """)

    with gr.Row():

        pdf_upload = gr.File(
            label="📄 Upload PDF",
            file_types=[".pdf"]
        )

        process_button = gr.Button(
            "⚙️ Process PDF",
            variant="primary"
        )

    document_info = gr.Markdown(
        "Upload a PDF and click **Process PDF**."
    )

    process_button.click(
        fn=process_pdf,
        inputs=pdf_upload,
        outputs=document_info
    )

    gr.Markdown("---")

    gr.Markdown("## 💬 Ask Questions")

    question = gr.Textbox(
        label="Your Question",
        placeholder="Example: What is the main objective of this document?"
    )

    top_k = gr.Slider(
        minimum=1,
        maximum=10,
        value=5,
        step=1,
        label="Number of chunks to retrieve"
    )

    ask_button = gr.Button(
        "🔎 ASK QUESTION",
        variant="primary"
    )

    answer = gr.Markdown(label="Answer")

    retrieved_chunks = gr.Markdown(
        label="Retrieved Chunks"
    )

    ask_button.click(
        fn=ask_question,
        inputs=[question, top_k],
        outputs=[answer, retrieved_chunks]
    )

    gr.Markdown("---")

    gr.Markdown("## 🧩 Inspect All Chunks")

    show_chunks_button = gr.Button(
        "📋 SHOW ALL CHUNKS"
    )

    all_chunks_output = gr.Markdown()

    show_chunks_button.click(
        fn=show_all_chunks,
        inputs=[],
        outputs=all_chunks_output
    )

In [27]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a6ec4441484b180375.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
